# RAG Demo with FAISS

This notebook implements a simple **Retrieval-Augmented Generation (RAG)** pipeline using FAISS for vector similarity search:

1. Load documents from CSV and create embeddings
2. Build a FAISS index for fast similarity search
3. For each query, retrieve the top-k most relevant documents

In [1]:
# Install dependencies if needed (uncomment and run once)
# %pip install pandas numpy faiss-cpu sentence-transformers

In [2]:
import os
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

/Users/larryjin/Documents/Programs/anaconda3/envs/prototype/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Paths to data (CSV files are in rag_data/)
DATA_DIR = "rag_data"
DOCS_PATH = os.path.join(DATA_DIR, "documents.csv")
QUERIES_PATH = os.path.join(DATA_DIR, "queries.csv")

# Load documents and queries
documents_df = pd.read_csv(DOCS_PATH)
queries_df = pd.read_csv(QUERIES_PATH)

print(f"Loaded {len(documents_df)} documents, {len(queries_df)} queries")
print("Document columns:", list(documents_df.columns))
documents_df.head(2)

Loaded 50 documents, 10 queries
Document columns: ['file_id', 'file_path', 'category', 'created_date', 'full_text']


,file_id,file_path,category,created_date,full_text
0,1,docs/python_basics.txt,programming,2024-01-15,Python is a high-level programming language kn...
1,2,docs/install_guide.txt,setup,2024-01-16,"To install Python on your system, download the..."


In [4]:
# Load a lightweight sentence embedding model (downloads on first run)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed all document texts
texts = documents_df["full_text"].astype(str).tolist()
doc_embeddings = model.encode(texts, show_progress_bar=True)
# doc_embeddings = np.array(doc_embeddings).astype("float32")
print(f"Embedding shape: {doc_embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21877.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 2/2 [00:00<00:00,  6.75it/s]

Embedding shape: (50, 384)


In [5]:
# Build FAISS index with L2 (Euclidean) distance — no normalization needed
dim = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(doc_embeddings)
print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")

FAISS index built: 50 vectors, dim=384


In [6]:
# Embed queries and search (raw embeddings; no normalization for L2 index)
TOP_K = 3  # number of documents to retrieve per query
query_texts = queries_df["query_text"].astype(str).tolist()
query_embeddings = model.encode(query_texts, show_progress_bar=True)
# query_embeddings = np.array(query_embeddings).astype("float32")

# Search: for each query, get top-k document indices (scores = L2 distances, lower is better)
scores, indices = index.search(query_embeddings, TOP_K)

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.34it/s]


In [7]:
# Display retrieval results for each query (scores = L2 distance, lower = more similar)
for i, row in queries_df.iterrows():
    qid = row["query_id"]
    query = row["query_text"]
    print(f"\n--- Query {qid}: {query}")
    for rank, (idx, dist) in enumerate(zip(indices[i], scores[i]), 1):
        doc = documents_df.iloc[idx]
        print(f"  [{rank}] (L2 dist={dist:.3f}) file_id={doc['file_id']} | {doc['file_path']}")
        print(f"      {doc['full_text'][:120]}...")


--- Query 1: How do I install Python on my computer?
  [1] (L2 dist=0.623) file_id=2 | docs/install_guide.txt
      To install Python on your system, download the installer from python.org. On Linux you can use apt-get install python3. ...
  [2] (L2 dist=0.936) file_id=1 | docs/python_basics.txt
      Python is a high-level programming language known for its readability and versatility. It supports multiple programming ...
  [3] (L2 dist=1.422) file_id=26 | docs/env_variables.txt
      Store config in environment variables. In Python use os.environ or python-dotenv. Never commit .env to git. Use differen...

--- Query 2: What is FAISS and how do I use it for similarity search?
  [1] (L2 dist=0.876) file_id=17 | docs/faiss_search.txt
      FAISS is a library for efficient similarity search. Build an index from vectors, then search for k nearest neighbors. Su...
  [2] (L2 dist=1.341) file_id=16 | docs/vector_embeddings.txt
      Embeddings map text or items to dense vectors. Similar ite

In [8]:
# Optional: helper to run RAG retrieval for any new query (e.g. to pass context to an LLM)
def retrieve(query: str, k: int = 3):
    """Return top-k documents for the given query. Score = L2 distance (lower is better)."""
    q_emb = model.encode([query])  # raw embeddings for L2 index
    q_emb = np.array(q_emb).astype("float32")
    distances, indices = index.search(q_emb, k)
    return [
        (documents_df.iloc[idx].to_dict(), float(distances[0][j]))
        for j, idx in enumerate(indices[0])
    ]

# Example: retrieve for a new query
example_query = "How to use Git for version control?"
results = retrieve(example_query, k=2)
for doc, dist in results:
    print(f"[L2={dist:.3f}] {doc['file_path']}: {doc['full_text'][:80]}...")

[L2=0.619] docs/version_control.txt: Version control tracks changes. Commit often with clear messages. Use branches f...
[L2=0.931] docs/env_variables.txt: Store config in environment variables. In Python use os.environ or python-dotenv...
